# ENEXRE - Full NER Training di Google Colab

Notebook ini menjalankan training NER BC5CDR menggunakan PubMedBERT dari repository `enexre`.

Sebelum menjalankan notebook:

1. Pilih `Runtime > Change runtime type > GPU`.
2. Push repository `enexre` ke GitHub.
3. Pastikan file processed NER tersedia di repository, atau siapkan file BC5CDR mentah agar preprocessing bisa dijalankan ulang.

Output utama akan disimpan ke `results/ner/`, `logs/ner/`, `predictions/ner/`, dan `checkpoints/ner/`.

## 1. Cek GPU

In [ ]:
!nvidia-smi

import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

## 2. Clone Repository dari GitHub

Ganti `GITHUB_REPO_URL` dengan URL repository kamu, misalnya `https://github.com/USERNAME/enexre.git`.

Untuk repository private, gunakan format `https://TOKEN@github.com/USERNAME/enexre.git`.

In [ ]:
#@title Konfigurasi repository
GITHUB_REPO_URL = "https://github.com/USERNAME/enexre.git"  #@param {type:"string"}
PROJECT_DIR = "/content/enexre"  #@param {type:"string"}

import os
import shutil
from pathlib import Path

project_path = Path(PROJECT_DIR)

if project_path.exists():
    print(f"Project directory already exists: {project_path}")
else:
    !git clone {GITHUB_REPO_URL} {PROJECT_DIR}

os.chdir(project_path)
print("Current directory:", Path.cwd())
!git status --short

## 3. Install Dependency

In [ ]:
!pip install -q -r requirements.txt

import torch
import transformers

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

## 4. Cek Dataset NER

Cell ini mengecek apakah dataset processed NER sudah tersedia. Jika belum tersedia, cell berikutnya bisa dipakai untuk membentuk ulang dataset dari `data/bc5cdr/*.txt`.

In [ ]:
from pathlib import Path

required_ner_files = [
    Path("data/processed/ner/train.jsonl"),
    Path("data/processed/ner/dev.jsonl"),
    Path("data/processed/ner/test.jsonl"),
    Path("data/processed/ner/label_map.json"),
]

for path in required_ner_files:
    print(path, "OK" if path.exists() else "MISSING")

all_processed_files_exist = all(path.exists() for path in required_ner_files)
print("Processed NER dataset ready:", all_processed_files_exist)

## 5. Opsional: Validasi dan Preprocessing Ulang

Jalankan cell ini hanya jika `data/processed/ner/*.jsonl` belum tersedia tetapi file BC5CDR mentah sudah ada di:

- `data/bc5cdr/train.txt`
- `data/bc5cdr/dev.txt`
- `data/bc5cdr/test.txt`

In [ ]:
RUN_PREPROCESSING = False  #@param {type:"boolean"}

if RUN_PREPROCESSING:
    !python scripts/validate_bc5cdr.py
    !python scripts/build_ner_dataset.py
else:
    print("Skip preprocessing. Set RUN_PREPROCESSING=True if processed NER files are missing.")

## 6. Smoke Test Training

Smoke test memastikan model, data loader, loss, evaluasi, dan penyimpanan metrik berjalan di Colab.

In [ ]:
RUN_SMOKE_TEST = True  #@param {type:"boolean"}

if RUN_SMOKE_TEST:
    !python scripts/train_ner.py --smoke-test --cleanup-smoke-checkpoint
else:
    print("Skip smoke test.")

## 7. Full Training Satu Konfigurasi

Gunakan cell ini untuk menjalankan satu training penuh. Secara default mengikuti konfigurasi eksperimen awal: seed 13, learning rate `1e-5`, batch size 8, maksimum 10 epoch, dan early stopping patience 2.

In [ ]:
RUN_SINGLE_FULL_TRAINING = True  #@param {type:"boolean"}
SEED = 13  #@param {type:"integer"}
LEARNING_RATE = "3e-5"  #@param ["1e-5", "3e-5", "5e-5"] {allow-input: true}
BATCH_SIZE = 8  #@param [8, 16] {type:"raw"}
EPOCHS = 10  #@param {type:"integer"}
PATIENCE = 2  #@param {type:"integer"}
RUN_NAME = "seed13_lr3e-5_bs8"  #@param {type:"string"}

if RUN_SINGLE_FULL_TRAINING:
    !python scripts/train_ner.py \
      --seed {SEED} \
      --learning-rate {LEARNING_RATE} \
      --batch-size {BATCH_SIZE} \
      --epochs {EPOCHS} \
      --patience {PATIENCE} \
      --run-name {RUN_NAME}
else:
    print("Skip single full training.")

## 8. Opsional: Seleksi Learning Rate

Cell ini menjalankan tiga learning rate dengan seed dan batch size yang sama. Gunakan setelah smoke test berhasil jika ingin memilih konfigurasi awal berdasarkan `best_dev_f1`.

In [ ]:
RUN_LR_SELECTION = False  #@param {type:"boolean"}
LR_SELECTION_SEED = 13  #@param {type:"integer"}
LR_SELECTION_BATCH_SIZE = 8  #@param [8, 16] {type:"raw"}

if RUN_LR_SELECTION:
    for lr in ["1e-5", "3e-5", "5e-5"]:
        run_name = f"seed{LR_SELECTION_SEED}_lr{lr}_bs{LR_SELECTION_BATCH_SIZE}"
        command = (
            f"python scripts/train_ner.py "
            f"--seed {LR_SELECTION_SEED} "
            f"--learning-rate {lr} "
            f"--batch-size {LR_SELECTION_BATCH_SIZE} "
            f"--run-name {run_name}"
        )
        print("Running:", command)
        exit_code = os.system(command)
        if exit_code != 0:
            raise RuntimeError(f"Training failed for {run_name}")
else:
    print("Skip learning-rate selection.")

## 9. Ringkas Hasil Training

Cell ini membaca semua file `results/ner/*_metrics.json` dan mengurutkan hasil berdasarkan `best_dev_f1`.

In [ ]:
import json
from pathlib import Path

metrics_rows = []
for path in sorted(Path("results/ner").glob("*_metrics.json")):
    data = json.loads(path.read_text(encoding="utf-8"))
    metrics_rows.append({
        "file": str(path),
        "run_name": data.get("run_name"),
        "seed": data.get("seed"),
        "learning_rate": data.get("learning_rate"),
        "batch_size": data.get("batch_size"),
        "best_epoch": data.get("best_epoch"),
        "best_dev_f1": data.get("best_dev_f1"),
        "checkpoint_dir": data.get("checkpoint_dir"),
    })

metrics_rows = sorted(metrics_rows, key=lambda row: row.get("best_dev_f1") or -1, reverse=True)

for row in metrics_rows:
    print(
        f"{row['best_dev_f1']:.4f} | {row['run_name']} | "
        f"seed={row['seed']} lr={row['learning_rate']} bs={row['batch_size']} "
        f"best_epoch={row['best_epoch']}"
    )

## 10. Evaluasi Model Terbaik pada Test Set

Cell ini memilih checkpoint terbaik berdasarkan `best_dev_f1` dari `results/ner/*_metrics.json`, lalu mengevaluasinya pada `data/processed/ner/test.jsonl`. Test set dipakai setelah model terbaik dipilih dari dev set.

In [ ]:
RUN_TEST_EVALUATION = True  #@param {type:"boolean"}
TEST_BATCH_SIZE = 16  #@param [8, 16, 32] {type:"raw"}
BEST_CHECKPOINT_DIR = ""  #@param {type:"string"}

import os

if RUN_TEST_EVALUATION:
    command = (
        "python scripts/evaluate_ner.py "
        f"--batch-size {TEST_BATCH_SIZE} "
        "--output results/ner/best_test_metrics.json "
        "--predictions predictions/ner/best_test_predictions.jsonl"
    )
    if BEST_CHECKPOINT_DIR.strip():
        command += f" --checkpoint {BEST_CHECKPOINT_DIR.strip()}"
    print("Running:", command)
    exit_code = os.system(command)
    if exit_code != 0:
        raise RuntimeError("NER test evaluation failed")
else:
    print("Skip test evaluation.")

## 11. Simpan Output ke Google Drive

Simpan hasil training agar tidak hilang saat runtime Colab berhenti.

In [ ]:
SAVE_TO_DRIVE = True  #@param {type:"boolean"}
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/enexre_outputs"  #@param {type:"string"}

if SAVE_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    output_dir = Path(DRIVE_OUTPUT_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)

    for source, target_name in [
        (Path("results/ner"), "results_ner"),
        (Path("logs/ner"), "logs_ner"),
        (Path("predictions/ner"), "predictions_ner"),
        (Path("checkpoints/ner"), "checkpoints_ner"),
    ]:
        target = output_dir / target_name
        if target.exists():
            shutil.rmtree(target)
        if source.exists():
            shutil.copytree(source, target)
            print(f"Copied {source} -> {target}")
        else:
            print(f"Skip missing source: {source}")
else:
    print("Skip saving to Drive.")

## 12. Catatan

- Jangan push `checkpoints/ner/` ke GitHub biasa kecuali memakai Git LFS.
- Untuk laporan eksperimen, catat `best_dev_f1`, `best_epoch`, seed, learning rate, batch size, dan checkpoint terbaik.
- Setelah model NER terbaik dipilih, lanjutkan ke tahap pembentukan kandidat Relation Extraction.